Constructed model inputs as: question + table headers (generate sql: ... [SEP] table: ...).


Tokenized inputs and SQL targets with t5-base tokenizer.

In [6]:
import json
import tarfile
from pathlib import Path

from transformers import T5TokenizerFast

PROJECT_ROOT = Path('.').resolve()
WIKISQL_DIR = PROJECT_ROOT / 'WikiSQL'
WIKISQL_ARCHIVE = WIKISQL_DIR / 'data.tar.bz2'
WIKISQL_DATA_DIR = WIKISQL_DIR / 'data'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 't5-base'

print('Project root:', PROJECT_ROOT)
print('WikiSQL dir:', WIKISQL_DIR)
print('WikiSQL archive:', WIKISQL_ARCHIVE)
print('WikiSQL data dir:', WIKISQL_DATA_DIR)
print('Processed dir:', PROCESSED_DIR)
print('Model:', MODEL_NAME)

tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)

Project root: /Users/srijareddy/Documents/GitHub/NL2SQL
WikiSQL dir: /Users/srijareddy/Documents/GitHub/NL2SQL/WikiSQL
WikiSQL archive: /Users/srijareddy/Documents/GitHub/NL2SQL/WikiSQL/data.tar.bz2
WikiSQL data dir: /Users/srijareddy/Documents/GitHub/NL2SQL/WikiSQL/data
Processed dir: /Users/srijareddy/Documents/GitHub/NL2SQL/data/processed
Model: t5-base


In [7]:
def _read_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _to_split_name(file_stem: str) -> str:
    if file_stem == 'dev':
        return 'validation'
    return file_stem


def load_wikisql_local():
    split_files = sorted(WIKISQL_DATA_DIR.glob('*.jsonl'))
    table_files = sorted(WIKISQL_DATA_DIR.glob('*.tables.jsonl'))

    if not split_files or not table_files:
        raise FileNotFoundError('error: data not found')

    table_map = {}
    for table_file in table_files:
        for table in _read_jsonl(table_file):
            table_map[table['id']] = {
                'id': table['id'],
                'header': table['header'],
                'types': table['types'],
            }

    dataset = {}
    for split_file in split_files:
        stem = split_file.stem
        if stem.endswith('.tables'):
            continue

        split = _to_split_name(stem)
        rows = _read_jsonl(split_file)
        examples = []

        for row in rows:
            table_id = row['table_id']
            if table_id not in table_map:
                continue

            sql_text = row.get('query')
            if sql_text is None:
                sql_text = json.dumps(row.get('sql', {}), ensure_ascii=True)

            examples.append({
                'question': row['question'],
                'sql': sql_text,
                'table_id': table_id,
                'table': table_map[table_id],
            })

        dataset[split] = examples

    print('loaded local WikiSQL splits:', {k: len(v) for k, v in dataset.items()})
    return dataset


def inspect_example(dataset, split='train', idx=0):
    ex = dataset[split][idx]
    print('Question:', ex['question'])
    print('SQL:', ex['sql'])
    print('Table ID:', ex['table_id'])
    print('Header:', ex['table']['header'])
    print('Types:', ex['table']['types'])
    return ex


wikisql = load_wikisql_local()
_ = inspect_example(wikisql, 'train', 0)

loaded local WikiSQL splits: {'validation': 8421, 'test': 15878, 'train': 56355}
Question: Tell me what the notes are for South Australia 
SQL: {"sel": 5, "conds": [[3, 0, "SOUTH AUSTRALIA"]], "agg": 0}
Table ID: 1-1000181-1
Header: ['State/territory', 'Text/background colour', 'Format', 'Current slogan', 'Current series', 'Notes']
Types: ['text', 'text', 'text', 'text', 'text', 'text']


In [8]:
def build_input_text(question: str, table: dict) -> str:
    header = table['header']
    header_str = ', '.join(header)
    return f"generate sql: {question} [SEP] table: {header_str}"


def preprocess_split(dataset_split, split_name: str, max_source_length: int = 256, max_target_length: int = 128):
    inputs = []
    targets = []

    for ex in dataset_split:
        src_text = build_input_text(ex['question'], ex['table'])
        tgt_text = ex['sql']
        inputs.append(src_text)
        targets.append(tgt_text)

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt',
    )['input_ids']

    labels[labels == tokenizer.pad_token_id] = -100

    model_inputs['labels'] = labels
    print(f"split={split_name}, num examples={len(inputs)}")
    return model_inputs


train_enc = preprocess_split(wikisql['train'], 'train')
val_enc = preprocess_split(wikisql['validation'], 'validation')

split=train, num examples=56355
split=validation, num examples=8421


In [9]:
import torch

train_path = PROCESSED_DIR / 'wikisql_t5_train.pt'
val_path = PROCESSED_DIR / 'wikisql_t5_val.pt'

torch.save(train_enc, train_path)
torch.save(val_enc, val_path)

print('Saved train to', train_path)
print('Saved val to', val_path)

Saved train to /Users/srijareddy/Documents/GitHub/NL2SQL/data/processed/wikisql_t5_train.pt
Saved val to /Users/srijareddy/Documents/GitHub/NL2SQL/data/processed/wikisql_t5_val.pt
